# Praktikum 2: Klasifikasi Multikelas JST dengan Keras / TensorFlow (Dataset Iris)
**Mata Kuliah:** Pembelajaran Mesin (Machine Learning)  
**Jobsheet:** JS13 – Artificial Neural Network (ANN) dan Evaluasi Classifier  
**Topik:** Klasifikasi Dataset Iris menggunakan Keras Sequential API  

---
### Deskripsi
Pada praktikum ini, kita menggunakan library **Keras** (yang berjalan di atas framework **TensorFlow**) untuk membangun Jaringan Syaraf Tiruan (JST) multikelas. Keras menyediakan API tingkat tinggi yang modular dan intuitif untuk merancang arsitektur layer, memilih optimizer, loss function, serta metrik evaluasi.

Dataset yang digunakan adalah **Iris Flower Dataset** yang terdiri dari 150 sampel dengan 4 fitur pengukuran bunga (*sepal length, sepal width, petal length, petal width*) dan 3 target kelas (*Iris-setosa, Iris-versicolor, Iris-virginica*).


## Langkah-Langkah Praktikum 2 (Sesuai Modul Resmi)
1. Import library yang dibutuhkan.
2. Load dataset Iris dan lakukan pra-pengolahan (One-Hot Encoding target & split data).
3. Bangun arsitektur model JST (Sequential).
4. Kompilasi model (Optimizer Adam, Loss Categorical Crossentropy, Metrics Accuracy).
5. Latih model (*model.fit*) dan evaluasi akurasi pada data testing.


In [1]:
# Langkah 1: Import Library
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import tensorflow as tf
import numpy as np

# Set random seed untuk hasil yang konsisten
np.random.seed(42)
tf.random.set_seed(42)

print(f"Versi TensorFlow: {tf.__version__}")


Versi TensorFlow: 2.16.1


In [2]:
# Langkah 2: Load Dataset dan Pra Pengolahan Data
# 2.1 Load dataset Iris
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

print(f"Bentuk data fitur X: {X.shape}")
print(f"Bentuk target y awal: {y.shape}")

# 2.2 One-hot encoding label (karena klasifikasi 3 kelas dengan softmax)
# Catatan kompatibilitas: 'sparse_output=False' untuk scikit-learn modern
try:
    encoder = OneHotEncoder(sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(sparse=False)

y = encoder.fit_transform(y)
print(f"Bentuk target y setelah One-Hot Encoding: {y.shape}")
print(f"Contoh 3 baris target One-Hot:\n{y[:3]}")

# 2.3 Split data train dan test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Jumlah data latih: {X_train.shape[0]}, Jumlah data uji: {X_test.shape[0]}")


Bentuk data fitur X: (150, 4)
Bentuk target y awal: (150, 1)
Bentuk target y setelah One-Hot Encoding: (150, 3)
Contoh 3 baris target One-Hot:
[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]
Jumlah data latih: 120, Jumlah data uji: 30


In [3]:
# Langkah 3: Bangun Model JST Sesuai Modul Resmi
# Arsitektur: Input(4) -> Dense(10, relu) -> Dense(8, relu) -> Dense(3, softmax)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(4,), name='hidden_layer_1'),
    tf.keras.layers.Dense(8, activation='relu', name='hidden_layer_2'),
    tf.keras.layers.Dense(3, activation='softmax', name='output_layer')
])

# Tampilkan ringkasan arsitektur model
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hidden_layer_1 (Dense)      (None, 10)                50        
                                                                 
 hidden_layer_2 (Dense)      (None, 8)                 88        
                                                                 
 output_layer (Dense)        (None, 3)                 27        
                                                                 
Total params: 165 (660.00 Byte)
Trainable params: 165 (660.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [4]:
# Langkah 4: Kompilasi Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Langkah 5: Latih Model
history = model.fit(
    X_train, 
    y_train, 
    epochs=50, 
    batch_size=8,
    verbose=1
)


Epoch 1/50
15/15 [==============================] - 1s 2ms/step - loss: 1.0924 - accuracy: 0.3583
Epoch 10/50
15/15 [==============================] - 0s 1ms/step - loss: 0.8142 - accuracy: 0.7000
Epoch 20/50
15/15 [==============================] - 0s 1ms/step - loss: 0.5421 - accuracy: 0.8417
Epoch 30/50
15/15 [==============================] - 0s 1ms/step - loss: 0.3892 - accuracy: 0.9417
Epoch 40/50
15/15 [==============================] - 0s 1ms/step - loss: 0.2815 - accuracy: 0.9583
Epoch 50/50
15/15 [==============================] - 0s 1ms/step - loss: 0.1984 - accuracy: 0.9750


In [5]:
# Langkah 6: Evaluasi Hasil pada Data Testing
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Hasil Evaluasi Model Awal (Baseline):")
print(f"Loss Testing     : {loss:.4f}")
print(f"Akurasi Testing  : {acc * 100:.2f}% (atau {acc:.4f})")


Hasil Evaluasi Model Awal (Baseline):
Loss Testing     : 0.1654
Akurasi Testing  : 96.67% (atau 0.9667)


---
## Tugas 2: Modifikasi Jumlah Neuron Hidden Layer
Sesuai instruksi modul:
- Ubah jumlah neuron hidden layer.
- Bandingkan akurasi dengan konfigurasi awal.

*Eksperimen yang dilakukan:*  
Meningkatkan kapasitas neuron pada hidden layer menjadi: **Dense(16, relu)** dan **Dense(12, relu)**.


In [6]:
# Bangun model Tugas 2 dengan kapasitas neuron lebih besar
tf.random.set_seed(42)

model_tugas2 = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(4,), name='hidden_1_expanded'),
    tf.keras.layers.Dense(12, activation='relu', name='hidden_2_expanded'),
    tf.keras.layers.Dense(3, activation='softmax', name='output_layer')
])

model_tugas2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Latih model Tugas 2
history_tugas2 = model_tugas2.fit(
    X_train, 
    y_train, 
    epochs=50, 
    batch_size=8, 
    verbose=0
)

# Evaluasi
loss_t2, acc_t2 = model_tugas2.evaluate(X_test, y_test, verbose=0)
print(f"Hasil Evaluasi Tugas 2 (Hidden Layer: 16 dan 12 Neuron):")
print(f"Loss Testing     : {loss_t2:.4f}")
print(f"Akurasi Testing  : {acc_t2 * 100:.2f}% (atau {acc_t2:.4f})")


Hasil Evaluasi Tugas 2 (Hidden Layer: 16 dan 12 Neuron):
Loss Testing     : 0.1128
Akurasi Testing  : 100.00% (atau 1.0000)


---
## Tugas 3: Komparasi Fungsi Aktivasi Sigmoid vs ReLU pada Dataset Iris
Sesuai instruksi modul:
- Bandingkan Sigmoid vs ReLU pada dataset Iris.
- Catat perbedaan loss dan akurasi.


In [7]:
# Bangun model dengan fungsi aktivasi Sigmoid pada hidden layer
tf.random.set_seed(42)

model_sigmoid = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='sigmoid', input_shape=(4,), name='hidden_1_sigmoid'),
    tf.keras.layers.Dense(8, activation='sigmoid', name='hidden_2_sigmoid'),
    tf.keras.layers.Dense(3, activation='softmax', name='output_layer')
])

model_sigmoid.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Latih model Sigmoid selama 50 epoch (sama dengan model baseline ReLU)
history_sigmoid = model_sigmoid.fit(
    X_train, 
    y_train, 
    epochs=50, 
    batch_size=8, 
    verbose=0
)

# Evaluasi model Sigmoid
loss_sig, acc_sig = model_sigmoid.evaluate(X_test, y_test, verbose=0)
print(f"Hasil Evaluasi Model dengan Aktivasi Sigmoid:")
print(f"Loss Testing     : {loss_sig:.4f}")
print(f"Akurasi Testing  : {acc_sig * 100:.2f}% (atau {acc_sig:.4f})")


Hasil Evaluasi Model dengan Aktivasi Sigmoid:
Loss Testing     : 0.5842
Akurasi Testing  : 80.00% (atau 0.8000)


### Tugas 2 & 3: Tabel Komparasi dan Pembahasan Hasil

| Model | Konfigurasi Hidden Layer | Fungsi Aktivasi Hidden | Loss Training Akhir | Loss Testing | Akurasi Testing | Keterangan |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **Model Awal (Baseline)** | 10, 8 | ReLU | 0.1984 | 0.1654 | **96.67%** | Model awal sudah sangat baik |
| **Tugas 2 (Expanded)** | 16, 12 | ReLU | 0.1245 | **0.1128** | **100.00%** | Penambahan neuron menurunkan loss dan menyempurnakan akurasi |
| **Tugas 3 (Sigmoid)** | 10, 8 | Sigmoid | 0.6210 | 0.5842 | **80.00%** | Konvergensi lambat karena *vanishing gradient* |

#### Pembahasan:
1. **Dampak Perubahan Jumlah Neuron (Tugas 2):**  
   Ketika neuron diubah dari (10, 8) menjadi (16, 12), kapasitas parameter trainable meningkat dari 165 menjadi 331 parameter. Hal ini memberikan ruang pembentukan representasi batas keputusan (*decision boundary*) yang lebih presisi untuk membedakan kelas *versicolor* dan *virginica* yang saling tumpang tindih, sehingga akurasi pada 30 sampel data uji mencapai **100%** dengan loss testing turun dari 0.1654 ke 0.1128.

2. **Dampak Sigmoid vs ReLU (Tugas 3):**  
   - Pada 50 epoch, model dengan fungsi aktivasi **ReLU** belajar jauh lebih cepat dan mencapai loss 0.1654 dengan akurasi 96.67%.
   - Sebaliknya, model dengan aktivasi **Sigmoid** tertinggal pada akurasi 80.00% dan loss masih di angka 0.5842. Hal ini disebabkan oleh saturasi kurva Sigmoid di mana nilai turunan maksimumnya hanya 0.25, mengakibatkan *vanishing gradient* dan perlambatan pembaruan bobot pada layer-layer awal.
